# DC-EGM for discrete-continuous choice

Notes at <https://dse.iskh.me/dcegm>.

## Discrete and continuous choice together

## The retirement model

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

class retirement():
    '''Consumption-savings model with a discrete retirement choice (Iskhakov et al. 2017)'''

    def __init__(self, beta=0.95, R=1.05, y=1.0, delta=0.5, T=10, sigma=0.0, sigma_y=0.0,
                 Mbar=20, ngrid=500, nquad=5):
        self.beta, self.R, self.y, self.delta = beta, R, y, delta   # discount, return, wage, disutility of work
        self.T, self.sigma, self.sigma_y = T, sigma, sigma_y  # periods, taste shock scale, income shock std
        self.Mbar, self.ngrid, self.nquad = Mbar, ngrid, nquad
        self.eps = np.finfo(float).eps                        # smallest positive float
        self.A = np.linspace(0, Mbar, ngrid)                  # grid on savings, ascending, starts at zero
        x, w = np.polynomial.hermite.hermgauss(nquad)         # Gauss-Hermite nodes and weights
        self.quadp = y * np.exp(np.sqrt(2) * sigma_y * x - sigma_y**2 / 2)   # income nodes, mean y
        self.quadw = w / np.sqrt(np.pi)                       # weights, sum to one

    def u(self, c, d):
        return np.log(c) - self.delta * d                     # utility, d = 1 working, d = 0 retired

    def mu(self, c):
        return 1 / c                                          # marginal utility

    def imu(self, x):
        return 1 / x                                          # inverse marginal utility

    def next_wealth(self, A, d, y):
        return self.R * A + d * y                             # budget: income only while working

def interp(x, xp, fp):
    '''Linear interpolation, extrapolated linearly beyond the last grid point'''
    slope = (fp[-1] - fp[-2]) / (xp[-1] - xp[-2])
    return np.where(x > xp[-1], fp[-1] + slope * (x - xp[-1]), np.interp(x, xp, fp))

## Choice-specific value functions

### The credit constrained region

In [ ]:
def terminal(m, d):
    '''Last period: everything is consumed, no endogenous points, the whole grid is the constrained region'''
    return {'M': np.array([]), 'c': np.array([]), 'v': np.array([]), 'ev0': 0.0}

def policy(m, s, M):
    '''Consumption at wealth M from the solution s of one period and choice: c = M below the first endogenous point'''
    M = np.asarray(M, float)
    if s['M'].size == 0: return M                             # last period: consume everything
    return np.where(M < s['M'][0], M, interp(M, s['M'], s['c']))

def value(m, s, d, M):
    '''Choice-specific value at wealth M: analytic below the first endogenous point, interpolated above'''
    M = np.asarray(M, float)
    vcc = m.u(np.maximum(M, m.eps), d) + m.beta * s['ev0']    # constrained region: c = M, A = 0
    if s['M'].size == 0: return vcc
    return np.where(M < s['M'][0], vcc, interp(M, s['M'], s['v']))

### The discrete choice

In [ ]:
def logsum(m, vW, vR):
    '''Expected value over the taste shocks: the maximum without shocks, the smoothed logsum with'''
    mx = np.maximum(vW, vR)
    if m.sigma == 0: return mx
    return mx + m.sigma * np.log(np.exp((vW - mx) / m.sigma) + np.exp((vR - mx) / m.sigma))   # de-maxed

def choice_prob(m, vW, vR):
    '''Probability to keep working: an indicator without taste shocks, logit with'''
    if m.sigma == 0: return (vW > vR).astype(float)
    eW, eR = np.exp((vW - np.maximum(vW, vR)) / m.sigma), np.exp((vR - np.maximum(vW, vR)) / m.sigma)
    return eW / (eW + eR)                                     # de-maxed, as in Class 6

### The EGM step for one discrete choice

In [ ]:
def egm_step(m, d, sol1):
    '''EGM step for choice d (1 work, 0 retire) given the solution sol1 = [retire, work] of next period'''
    M1 = m.next_wealth(m.A[:, None], d, m.quadp[None, :])     # next period wealth: savings x income nodes
    vR = value(m, sol1[0], 0, M1)                             # next period value of a retiree
    cR = np.maximum(policy(m, sol1[0], M1), m.eps)            # next period consumption of a retiree, kept positive
    if d == 1:                                                # still a worker tomorrow: chooses again
        vW = value(m, sol1[1], 1, M1)
        cW = np.maximum(policy(m, sol1[1], M1), m.eps)
        P = choice_prob(m, vW, vR)                            # probability to keep working tomorrow
        mu1 = P * m.mu(cW) + (1 - P) * m.mu(cR)               # expected marginal utility over the choice
        V1 = logsum(m, vW, vR)                                # value before the taste shocks are drawn
    else:                                                     # retired tomorrow: no choice
        mu1, V1 = m.mu(cR), vR
    rhs = m.beta * m.R * (mu1 @ m.quadw)                      # RHS of the Euler equation, by quadrature
    c = m.imu(rhs)                                            # inverted Euler equation
    M = m.A + c                                               # endogenous grid points
    ev = V1 @ m.quadw                                         # expected value of next period at each A
    v = m.u(c, d) + m.beta * ev                               # choice-specific value at the endogenous points
    return M, c, v, ev[0]                                     # ev[0]: expected value of saving nothing

In [ ]:
m = retirement()
sol = [[terminal(m, d) for d in (0, 1)]]                      # last period, index T-1
for t in range(2):
    sol.insert(0, [dict(zip(('M', 'c', 'v', 'ev0'), egm_step(m, d, sol[0]))) for d in (0, 1)])   # T-2, then T-3
Mg = np.linspace(0.05, m.Mbar, 1000)                          # dense grid for the plots
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for s, lab in ((sol[2], 'last period'), (sol[1], 'one before last')):
    for d, name in ((0, 'retire'), (1, 'work')):
        ax[0].plot(Mg, value(m, s[d], d, Mg), label=f'{lab}, {name}')
vW, vR = value(m, sol[1][1], 1, Mg), value(m, sol[1][0], 0, Mg)
c = np.where(vW > vR, policy(m, sol[1][1], Mg), policy(m, sol[1][0], Mg))   # the worker's consumption
ax[1].plot(Mg, c, label='worker: retire or work')
ax[1].plot(Mg, policy(m, sol[1][0], Mg), label='retiree')
ax[0].set(ylim=(-3, 5), xlabel='wealth M', title='Choice-specific value functions of a worker')
ax[1].set(ylim=(0, 10.5), xlabel='wealth M', title='Consumption one period before the last')
for a in ax: a.grid(color='0.9'); a.legend()
plt.show()

## Why the Euler equation is not enough

In [ ]:
def euler_residual(m, sol1, M, c):
    '''Euler residual of a worker who keeps working, at wealth M and consumption c, given next period solution'''
    M1 = m.next_wealth(np.asarray(M)[..., None] - np.asarray(c)[..., None], 1, m.quadp)   # ... x nodes
    vW, vR = value(m, sol1[1], 1, M1), value(m, sol1[0], 0, M1)
    P = choice_prob(m, vW, vR)                                # who works next period
    cW, cR = np.maximum(policy(m, sol1[1], M1), m.eps), np.maximum(policy(m, sol1[0], M1), m.eps)
    return m.mu(c) - m.beta * m.R * ((P * m.mu(cW) + (1 - P) * m.mu(cR)) @ m.quadw)

In [ ]:
M, c, v, ev0 = egm_step(m, 1, sol[1])                         # raw EGM step of the worker, two periods before last
bend = np.flatnonzero(np.diff(M) <= 0)                        # steps where the endogenous grid bends back
lo, hi = M[bend].min() - 0.3, M[bend].max() + 0.3             # window around the zig-zag
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].plot(M, v, '.-', ms=4, lw=0.7)
ax[1].plot(M, c, '.-', ms=4, lw=0.7)
ax[0].set(xlim=(lo, hi), ylim=(np.interp(lo, M, v) - 0.03, np.interp(hi, M, v) + 0.03),
          xlabel='wealth M', title='Value at the EGM points, in savings order')
ax[1].set(xlim=(lo, hi), ylim=(np.interp(lo, M, c) - 0.3, np.interp(hi, M, c) + 0.3),
          xlabel='wealth M', title='Consumption at the EGM points')
cg = np.linspace(1.6, 2.6, 500)
for Mz in np.linspace(lo + 0.1, hi - 0.1, 5):
    ax[2].plot(cg, euler_residual(m, sol[1], Mz, cg), lw=0.8, label=f'M = {Mz:.2f}')
ax[2].axhline(0, color='k', lw=0.5)
ax[2].set(ylim=(-0.15, 0.15), xlabel='consumption c', title='Euler residuals: several zeros'); ax[2].legend()
for a in ax: a.grid(color='0.9')
plt.show()

## The secondary upper envelope

In [ ]:
def segments(M):
    '''Indices of the maximal monotone runs of M, each run ascending in M, the turning points shared'''
    up = np.diff(M) > 0                                       # direction of every step
    turn = np.flatnonzero(up[1:] != up[:-1]) + 1              # steps where the direction changes
    bounds = np.r_[0, turn, M.size - 1]
    runs = [np.arange(a, b + 1) for a, b in zip(bounds[:-1], bounds[1:])]
    return [r if M[r[-1]] > M[r[0]] else r[::-1] for r in runs]   # flip the descending runs

In [ ]:
def envelope_lines(m, d, M, c, v, ev0, runs):
    '''Every run on the wealth points inside its own interval, the envelope, and the line on top at every point'''
    x = np.sort(M, kind='stable')                             # all wealth points; a stable sort merges the sorted runs
    x = x[np.r_[True, np.diff(x) > 0]]                        # drop the duplicates: the shared turning points
    ncc = np.searchsorted(x, M[0])                            # points below the first endogenous point, if any
    lines = [(0, ncc, m.u(x[:ncc], d) + m.beta * ev0, x[:ncc])]   # line 0: credit constrained, c = M, analytic value
    for r in runs:
        lo, hi = np.searchsorted(x, M[r[0]]), np.searchsorted(x, M[r[-1]], 'right')   # x-points inside the run
        lines.append((lo, hi, np.interp(x[lo:hi], M[r], v[r]), np.interp(x[lo:hi], M[r], c[r])))
    vmax, top = np.full(x.size, -np.inf), np.full(x.size, -1) # the envelope and the line on top at every point
    for i, (lo, hi, V, C) in enumerate(lines):
        better = V > vmax[lo:hi]                              # where this line beats the envelope so far
        vmax[lo:hi] = np.where(better, V, vmax[lo:hi])
        top[lo:hi] = np.where(better, i, top[lo:hi])
    return x, lines, vmax, top

In [ ]:
def crossing(m, d, ev0, x, lines, a, b, j):
    '''Kink where line b overtakes line a on [x[j], x[j+1]]: wealth, value, consumption left and right'''
    (lo0, hi0, V0, C0), (lo1, hi1, V1, C1) = lines[a], lines[b]
    if not (lo0 <= j < hi0 - 1 and lo1 <= j < hi1 - 1): return None   # a line ends or starts here: no crossing
    f0, f1 = V1[j - lo1] - V0[j - lo0], V1[j + 1 - lo1] - V0[j + 1 - lo0]   # difference of the lines at both ends
    if f0 * f1 >= 0: return None                              # no sign change: the lines share the point
    at = lambda A, lo, w: A[j - lo] + w * (A[j + 1 - lo] - A[j - lo])   # a line evaluated at x[j] + w (x[j+1] - x[j])
    w = f0 / (f0 - f1)                                        # both lines straight on [x[j], x[j+1]]: closed form
    z, vz = x[j] + w * (x[j + 1] - x[j]), at(V0, lo0, w)       # the kink and the value there
    if a == 0 or b == 0:                                      # the constrained curve is concave: refine by a root finder
        lo, V = (lo1, V1) if a == 0 else (lo0, V0)            # the run that crosses it
        g = lambda z: m.u(z, d) + m.beta * ev0 - at(V, lo, (z - x[j]) / (x[j + 1] - x[j]))
        z = brentq(g, x[j], x[j + 1])
        w, vz = (z - x[j]) / (x[j + 1] - x[j]), m.u(z, d) + m.beta * ev0
    return z, vz, at(C0, lo0, w), at(C1, lo1, w)              # consumption jumps at the kink: left line, right line

In [ ]:
def upper_envelope(m, d, M, c, v, ev0):
    '''Delete the endogenous points below the upper envelope of the curve (M, v), insert the kinks'''
    runs = segments(M)
    if len(runs) == 1: return M, c, v                         # no bend-back: nothing to do
    x, lines, vmax, top = envelope_lines(m, d, M, c, v, ev0, runs)
    vM = vmax[np.searchsorted(x, M)]                          # the envelope at the original points
    keep = v >= vM - 1e-10 * (1 + np.abs(vM))                 # keep the points on the envelope only
    Mn, cn, vn = [M[keep]], [c[keep]], [v[keep]]
    for j in np.flatnonzero(np.diff(top)):                    # the top line changes between x[j] and x[j+1]
        a, z = top[j], x[j]                                   # walk from the line on top at x[j] ...
        while a != top[j + 1]:                                # ... to the line on top at x[j+1]
            first = None                                      # the first line to overtake a after z
            for b in range(len(lines)):
                k = None if b == a else crossing(m, d, ev0, x, lines, a, b, j)
                if k is not None and k[0] > z and (first is None or k[0] < first[0][0]): first = (k, b)
            if first is None: break                           # the lines only share a point here
            (z, vz, c0, c1), a = first
            Mn.append([z - 1e3 * m.eps, z]); vn.append([vz, vz]); cn.append([c0, c1])   # the kink, twice
    Mn, cn, vn = map(np.concatenate, (Mn, cn, vn))
    i = np.argsort(Mn, kind='stable')                         # nearly sorted already: linear time
    return Mn[i], cn[i], vn[i]

In [ ]:
Mc, cc, vc = upper_envelope(m, 1, M, c, v, ev0)             # the raw points of the figure above, cleaned
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(M, v, '.-', ms=4, lw=0.7, color='0.6', label='EGM points, in savings order')
ax[0].plot(Mc, vc, 'o-', ms=3, lw=1, color='C3', label='after the upper envelope')
ax[1].plot(M, c, '.-', ms=4, lw=0.7, color='0.6', label='EGM points, in savings order')
ax[1].plot(Mc, cc, 'o-', ms=3, lw=1, color='C3', label='after the upper envelope')
ax[0].set(xlim=(lo, hi), ylim=(np.interp(lo, M, v) - 0.03, np.interp(hi, M, v) + 0.03),
          xlabel='wealth M', title='Value: the upper envelope and the secondary kink')
ax[1].set(xlim=(lo, hi), ylim=(np.interp(lo, M, c) - 0.3, np.interp(hi, M, c) + 0.3),
          xlabel='wealth M', title='Consumption: the jump at the secondary kink')
for a in ax: a.grid(color='0.9'); a.legend()
print(f'{M.size} endogenous points in, {Mc.size} out: {np.sum(~np.isin(Mc, M))} inserted at the kink')
plt.show()

## The full algorithm without taste shocks

In [ ]:
def solve_dcegm(m, callback=None):
    '''Backward induction with DC-EGM: returns sol[t][d], t = 0..T-1, d = 0 retire, 1 work'''
    sol = [[None, None] for t in range(m.T)]
    sol[-1] = [terminal(m, d) for d in (0, 1)]
    for t in range(m.T - 2, -1, -1):
        for d in (0, 1):
            M, c, v, ev0 = egm_step(m, d, sol[t + 1])
            if d == 1:                                        # only the worker's curve can bend back
                M, c, v = upper_envelope(m, d, M, c, v, ev0)
            sol[t][d] = {'M': M, 'c': c, 'v': v, 'ev0': ev0}
            assert np.all(np.diff(M) > 0)                     # the cleaned endogenous grid is sorted
            if callback: callback(t=t, d=d, sol=sol[t][d])
    return sol

def worker(m, sol_t, M):
    '''Policy of a worker in one period: consumption, the probability to work, and the value'''
    vW, vR = value(m, sol_t[1], 1, M), value(m, sol_t[0], 0, M)
    P = choice_prob(m, vW, vR)
    c = np.where(P > 0.5, policy(m, sol_t[1], M), policy(m, sol_t[0], M))   # the more likely choice
    return c, P, logsum(m, vW, vR)

In [ ]:
t0 = time.perf_counter()
sol = solve_dcegm(m)
print(f'DC-EGM solved {m.T} periods on {m.ngrid} grid points in {time.perf_counter() - t0:.3f} seconds')
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for t, col in zip(range(m.T - 1), plt.cm.viridis(np.linspace(0, 0.9, m.T - 1))):
    c, P, V = worker(m, sol[t], Mg)
    ax[0].plot(Mg, c, color=col, lw=0.9, label=f't = {t}')
    ax[1].plot(Mg, V, color=col, lw=0.9)
    ax[2].plot(Mg, P, color=col, lw=0.9)
ax[0].set(ylim=(0, 8), xlabel='wealth M', title="Worker's consumption, t = 0 (dark) to T-2 (light)")
ax[1].set(ylim=(-4, 10), xlabel='wealth M', title="Worker's value function")
ax[2].set(xlabel='wealth M', title='Decision to keep working')
ax[0].legend(fontsize=8)
for a in ax: a.grid(color='0.9')
plt.show()

### Check against the analytic solution

In [ ]:
ma = retirement(R=1.0)                                        # log utility, no interest: analytic case
sa = solve_dcegm(ma)
S = lambda k: sum(ma.beta**i for i in range(k + 1))           # S_k = 1 + beta + ... + beta^k
for t in (0, 4, 7, 8):
    tg = ma.T - 1 - t                                         # periods to go
    c, P, V = worker(ma, sa[t], Mg)
    lines = [Mg] + [(k * ma.y + Mg) / S(k) for k in range(1, tg)] + [(k * ma.y + Mg) / S(tg) for k in range(tg + 1)]
    dev = np.min(np.abs(np.array(lines) - c), axis=0)         # distance to the nearest analytic line
    print(f'{tg} periods to go: {np.mean(dev < 1e-10) * 100:5.1f}% of the grid points within 1e-10 of the '
          f'analytic policy, max deviation {dev.max():.1e} at M = {Mg[dev.argmax()]:.2f}')

## Random income

In [ ]:
t = m.T - 4
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for sy, nq, ls in ((0, 5, '-'), (0.2, 5, '-'), (0.5, 5, '-'), (0.5, 30, ':')):
    ms = retirement(sigma_y=sy, nquad=nq)
    c, P, V = worker(ms, solve_dcegm(ms)[t], Mg)
    ax[0].plot(Mg, c, ls, lw=1, label=f'σ_y = {sy}, {nq} nodes')
    ax[1].plot(Mg, P, ls, lw=1)
ax[0].set(xlim=(2, 9), ylim=(1.3, 2.6), xlabel='wealth M', title=f'Consumption of a worker, t = {t}')
ax[1].set(xlabel='wealth M', title='Decision to keep working')
ax[0].legend()
for a in ax: a.grid(color='0.9')
plt.show()

## Extreme value taste shocks

### The smoothed Euler equation

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
Mz = 0.5 * (lo + hi)                                          # a wealth level inside the zig-zag region
for s in (0, 0.01, 0.05, 0.1):
    ms = retirement(sigma=s)
    ss = solve_dcegm(ms)
    c, P, V = worker(ms, ss[t], Mg)
    ax[0].plot(Mg, policy(ms, ss[t][1], Mg), lw=1, label=f'σ = {s}')   # consumption conditional on working
    ax[1].plot(Mg, P, lw=1)
    ax[2].plot(cg, euler_residual(ms, ss[m.T - 2], Mz, cg), lw=1)
ax[0].set(xlim=(2, 9), ylim=(1.3, 2.6), xlabel='wealth M', title=f'Consumption of a worker who keeps working, t = {t}')
ax[1].set(xlabel='wealth M', title='Probability to keep working')
ax[2].axhline(0, color='k', lw=0.5)
ax[2].set(ylim=(-0.15, 0.15), xlabel='consumption c', title=f'Euler residual at M = {Mz:.2f}, t = T-3')
ax[0].legend()
for a in ax: a.grid(color='0.9')
plt.show()

## Simulation

In [ ]:
def simulate(m, sol, M0, seed=None):
    '''Simulate the life cycle of workers from initial wealth M0 (array): wealth, consumption, work status'''
    rng = np.random.default_rng(seed)
    N = np.atleast_1d(M0).size
    M, C, D = np.empty((N, m.T)), np.empty((N, m.T)), np.ones((N, m.T), int)
    M[:, 0] = M0
    for t in range(m.T):
        if t > 0: D[:, t] = D[:, t - 1]                       # retirement is absorbing
        w = D[:, t] == 1                                      # those still working decide
        vW, vR = value(m, sol[t][1], 1, M[w, t]), value(m, sol[t][0], 0, M[w, t])
        D[w, t] = rng.random(w.sum()) < choice_prob(m, vW, vR)   # taste shocks, through the logit
        for d in (0, 1):
            i = D[:, t] == d
            C[i, t] = policy(m, sol[t][d], M[i, t])           # consumption conditional on the choice
        if t < m.T - 1:
            y = m.y * np.exp(m.sigma_y * rng.standard_normal(N) - m.sigma_y**2 / 2)   # income draws
            M[:, t + 1] = m.next_wealth(M[:, t] - C[:, t], D[:, t], y)
    return M, C, D

In [ ]:
ms = retirement(sigma=0.1, sigma_y=0.2, nquad=10)
ss = solve_dcegm(ms)
M0 = np.random.default_rng(1).uniform(2, 20, 1000)            # initial wealth of 1000 workers
Ms, Cs, Ds = simulate(ms, ss, M0, seed=2026)
age = np.where((Ds == 0).any(axis=1), np.argmin(Ds, axis=1), ms.T)   # first period of retirement
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for i in range(15):
    ax[0].plot(Ms[i], lw=0.8, color='C0'); ax[1].plot(Cs[i], lw=0.8, color='C1')
    if age[i] < ms.T:
        ax[0].plot(age[i], Ms[i, age[i]], 'ks', ms=4); ax[1].plot(age[i], Cs[i, age[i]], 'ks', ms=4)
ax[2].hist(age, bins=np.arange(ms.T + 2) - 0.5, color='C2')
ax[0].set(xlabel='period', title='Wealth of 15 simulated workers, square: retirement')
ax[1].set(xlabel='period', title='Consumption')
ax[2].set(xlabel='first period of retirement', title='Distribution of the retirement age, 1000 workers')
for a in ax: a.grid(color='0.9')
plt.show()

## Keane's test

Deterministic model, no taste shocks, no income shocks, $\beta R = 1$. The utility is
additively separable in consumption and work, so the Euler equation holds across the
retirement decision too: on any path where the credit constraint does not bind,
consumption is the same in every period, before and after retirement.

In [ ]:
mk = dict(R=1 / 0.95, T=20, Mbar=40)                          # beta R = 1, sigma = sigma_y = 0 by default
M0 = np.linspace(1, 20, 191)                                  # initial wealth of the simulated workers
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for ngrid in (100, 500, 2000):
    mt = retirement(ngrid=ngrid, **mk)
    st = solve_dcegm(mt)
    Mt, Ct, Dt = simulate(mt, st, M0)
    free = (Mt[:, :-1] - Ct[:, :-1] > 0).all(axis=1)          # savings positive: the constraint never binds
    dev = np.abs(Ct / Ct[:, :1] - 1).max(axis=1)              # largest deviation from the initial consumption
    ax[1].semilogy(M0[free], np.maximum(dev[free], 1e-16), '.-', ms=3, lw=0.7, label=f'{ngrid} grid points')
    print(f'{ngrid} grid points: {free.sum()} of {M0.size} paths unconstrained, '
          f'max |c_t / c_0 - 1| = {dev[free].max():.1e}')
    if ngrid == 500:
        age = np.where((Dt == 0).any(axis=1), np.argmin(Dt, axis=1), mt.T)   # first period of retirement
        for i in range(0, M0.size, 19):
            ax[0].plot(Ct[i], lw=0.8, color='C0')
            if age[i] < mt.T - 1: ax[0].plot(age[i], Ct[i, age[i]], 'ks', ms=4)
ax[0].set(xlabel='period', title='Consumption of simulated workers, 500 grid points, square: retirement')
ax[1].set(xlabel='initial wealth M0', title="Keane's test, max |c_t / c_0 - 1| on unconstrained paths")
for a in ax: a.grid(color='0.9')
ax[1].legend()
plt.show()